In [183]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [184]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

import torch
import torch_geometric as pyg
import mrmr

from baseline_evals.feature_selection import variance_filtering

from torch_geometric.utils import to_networkx


from bipartite_gnn.train_test import GNNTrainer
from bipartite_gnn.model import GAT_2L, BipartiteRGAT, BiRGAT
from bipartite_gnn.graph_building import cosine_similarity_matrix, dense_to_coo
from bipartite_gnn.preprocessing import gg_interactions, get_mirna_gene_interactions, pp_interactions, get_interactions
from bipartite_gnn.graph_building import create_diff_exp_connections_nbnom, create_diff_exp_connections_norm, dense_to_coo
from baseline_evals.feature_selection import class_variational_selection, variance_filtering
from bipartite_gnn.preprocessing import mrmr_selection
import torch_geometric.transforms as T
from bipartite_gnn.graph_building import dense_to_attributes

torch.set_printoptions(sci_mode=False)

In [185]:
# allrnas
allrna = pl.read_csv('MDS_data/200625_allRNA_fromRNAseq_annot_hg38.tsv', separator='\t')
allrna_gene_names = allrna['GENE_NAME'].to_list()
alrnna_geme_id = allrna['GENE_ID'].to_list()
# allrna = allrna[:, 6:]

mrna = allrna.filter(
    allrna['GENE_TYPE'].is_in(["protein_coding"])
)

mrna_names = mrna['GENE_NAME'].to_list()

# mirna
mirna = allrna.filter(
    allrna['GENE_TYPE'].is_in(["miRNA"])
)

mirna_names = mirna['GENE_NAME'].to_list()

for i in range(len(mirna_names)):
    mirna_names[i] = "hsa-mir-" + mirna_names[i][3:]

# pirna
pirna_counts = pl.read_excel(
    "MDS_data/piRNA_counts.xlsx"
)
# remove low variance rows
pirna_counts = pirna_counts[350:]

# circrna
circrna = pl.read_csv(
    "MDS_data/200625_circRNA_fromRNAseq_annot_hg19.tsv",
    separator='\t',
    null_values=['NA']
)
circrna = circrna.drop_nulls() # lots of missing data in circrna
circrna_id = circrna['circRNA_ID'].to_list()
circrna_gene_names = circrna['GENE_NAME'].to_list()

# te counts
te_counts = pl.read_csv(
    'MDS_data/TE_counts.csv',
    separator=','
)

circrna_mirna_interactions = pl.read_csv(
    "MDS_data/circrna-mirna-interactions-mirbase.csv",
    separator=','
)

mirna_mrna_interactions = pl.read_csv(
    'MDS_data/mirna-mrna-interactions.csv',
    separator=','
)

annot = pl.read_excel(
    'MDS_data/sample sheet for CVUT.xlsx',
)

annot = annot.select([
    "SAMPLE_NAME", '1 disease', '2 risk', '3 mutations (SF3B1only_wt)'
])

# notable rnas from allrna
- protein_coding
- miRNA
- processed_pseudogene, can have regulatory functions
- antisense, can have regulatory functions
- lincRNA, "transcriptional noise ?"

In [186]:
annotated_samples = annot['SAMPLE_NAME'].to_list()

sample_ids = [name.split('_')[0] for name in annotated_samples]
sample_ids_set = set(sample_ids)

annot = annot.with_columns(sample_ids=pl.Series(sample_ids))

In [187]:
allrna_names = allrna.columns[6:]
allrna_ids = [name.split('_')[0] for name in allrna_names]

mrna.columns = mrna.columns[:6] + allrna_ids
mirna.columns = mirna.columns[:6] + allrna_ids

te_sample_names = te_counts.columns[1:]
te_sample_ids = [name.split('_')[0] for name in te_sample_names]

te_counts.columns = te_counts.columns[:1] + te_sample_ids

circna_sample_names = circrna.columns[9:]
circrna_sample_ids = [name.split('_')[0] for name in circna_sample_names]

circrna.columns = circrna.columns[:9] + circrna_sample_ids

In [188]:
pirna_cols = pirna_counts.columns
pirna_sample_names = pirna_counts.columns[1:]
pirna_sample_ids = [name.split('_')[0] for name in pirna_sample_names]

In [189]:
# excluding pirna we get 8 more samples
# pirnas are also measure from different samples
# than the other expression data, which poses questions
# whether the expression data is comparable across samples
common_names = sample_ids_set \
    .intersection(set(allrna_ids)) \
    .intersection(set(te_sample_ids)) \
    .intersection(set(circrna_sample_ids)) \
    # .intersection(set(pirna_sample_ids))

common_names = list(common_names)

len(common_names)

74

In [190]:
# prepared data
mrna = mrna.select(['GENE_NAME'] + common_names)
mirna = mirna.select(['GENE_NAME'] + common_names)
circrna = circrna.select(['circRNA_ID'] + common_names)
te_counts = te_counts.select(['TE'] + common_names)

In [191]:
mrna.shape, mirna.shape, circrna.shape, te_counts.shape

((19873, 75), (1879, 75), (69, 75), (687, 75))

In [192]:
def order_col_by_list(df, col_name, custom_order):
    # Create a mapping from the custom list to a numerical order
    order_mapping = {value: index for index, value in enumerate(custom_order)}

    # Apply the mapping to the 'name' column to create a new column with the numerical order
    df = df.with_columns(
        pl.col(col_name)
            .apply(lambda x: order_mapping[x], return_dtype=pl.Int32)
            .alias("order")
    )

    # Sort the DataFrame based on the 'order' column
    df = df.sort("order")

    # Optionally, drop the 'order' column if it's no longer needed
    df = df.drop("order")

    return df

# sort annotations in the same order as the expression data
annot = order_col_by_list(
    annot.filter(annot['sample_ids'].is_in(common_names)),
    'sample_ids', 
    common_names
)

In [193]:
# ensure that samples and target values are aligned
assert \
    annot['sample_ids'].to_list() ==\
    mrna.columns[1:] ==\
    mirna.columns[1:] ==\
    circrna.columns[1:] ==\
    te_counts.columns[1:]

In [194]:
# classes and class counts
y_disease = annot['1 disease'].to_numpy() - 1
y_risk = annot['2 risk'].to_numpy()
y_mutations = annot['3 mutations (SF3B1only_wt)'].to_numpy()

# select target variable
y = y_mutations

np.unique(y_disease, return_counts=True), np.unique(y_risk, return_counts=True), np.unique(y_mutations, return_counts=True)

((array([0, 1]), array([13, 61])),
 (array([0, 1, 2]), array([21, 29, 24])),
 (array([0, 1, 2]), array([48, 14, 12])))

In [195]:
# write to disk
mrna.write_csv('MDS_data_preprocessed/mrna.csv')
mirna.write_csv('MDS_data_preprocessed/mirna.csv')
circrna.write_csv('MDS_data_preprocessed/circrna.csv')
te_counts.write_csv('MDS_data_preprocessed/te_counts.csv')
annot.write_csv('MDS_data_preprocessed/annotations.csv')

In [196]:
# mrna_X = torch.tensor(mrna[:, 1:].to_numpy().T)
# mirna_X = torch.tensor(mirna[:, 1:].to_numpy().T)
# circrna_X = torch.tensor(circrna[:, 1:].to_numpy().T)
# te_X = torch.tensor(te_counts[:, 1:].to_numpy().T)

random_state = 3

# split training and testing data
train_mask, test_mask = train_test_split(torch.arange(len(y)), test_size=0.3, random_state=random_state, stratify=y)
val_mask, test_mask = train_test_split(test_mask, test_size=0.5, random_state=random_state, stratify=y[test_mask])

np.unique(y[train_mask], return_counts=True), np.unique(y[val_mask], return_counts=True), np.unique(y[test_mask], return_counts=True)

((array([0, 1, 2]), array([33, 10,  8])),
 (array([0, 1, 2]), array([7, 2, 2])),
 (array([0, 1, 2]), array([8, 2, 2])))

In [197]:
len(set(te_counts['TE'].to_list())), len(te_counts['TE'].to_list())

(687, 687)

In [198]:
len(set(circrna['circRNA_ID'].to_list())), len(circrna['circRNA_ID'].to_list())

(69, 69)

In [199]:
mrna_X_s = mrmr_selection(
    X_df=mrna,
    y=y,
    train_mask=train_mask,
    n_features=500,
    n_preselected_features=2000,
    feature_col_name='GENE_NAME',
)
mirna_X_s = mrmr_selection(
    X_df=mirna,
    y=y,
    train_mask=train_mask,
    n_features=500,
    n_preselected_features=2000,
    feature_col_name='GENE_NAME',
)
circrna_X_s = mrmr_selection(
    X_df=circrna,
    y=y,
    train_mask=train_mask,
    n_features=50,
    n_preselected_features=None,
    feature_col_name='circRNA_ID',
)
te_X_s = mrmr_selection(
    X_df=te_counts,
    y=y,
    train_mask=train_mask,
    n_features=250,
    n_preselected_features=None,
    feature_col_name='TE',
)

100%|██████████| 250/250 [00:21<00:00, 11.40it/s]


In [245]:
mrna_X = torch.tensor(mrna_X_s[:, 1:].to_numpy(), dtype=torch.float32).T
mrna_features = mrna_X_s[:, 0].to_list()
mirna_X = torch.tensor(mirna_X_s[:, 1:].to_numpy(), dtype=torch.float32).T
mirna_features = mirna_X_s[:, 0].to_list()
circrna_X = torch.tensor(circrna_X_s[:, 1:].to_numpy(), dtype=torch.float32).T
circrna_features = circrna_X_s[:, 0].to_list()
te_X = torch.tensor(te_X_s[:, 1:].to_numpy(), dtype=torch.float32).T
te_features = te_X_s[:, 0].to_list()

In [246]:
mirna_features = mirna_X_s['GENE_NAME'].to_list()

for i in range(len(mirna_features)):
    mirna_features[i] = "hsa-mir-" + mirna_features[i][3:]

mrna_features = mrna_X_s['GENE_NAME'].to_list()

circrna_features = circrna_X_s['circRNA_ID'].to_list()

te_features = te_X_s['TE'].to_list()

In [201]:
# normalize in each patient
# mrna_X = mrna_X / mrna_X.sum(axis=0)
# mirna_X = mirna_X / mirna_X.sum(axis=0)
# circrna_X = circrna_X / circrna_X.sum(axis=0)
# te_X = te_X / te_X.sum(axis=0)

# # log transform
# mrna_X = np.log2(mrna_X + 1).T
# mirna_X = np.log2(mirna_X + 1).T
# circrna_X = np.log2(circrna_X + 1).T
# te_X = np.log2(te_X + 1).T

In [247]:
mrna_A = create_diff_exp_connections_norm(mrna_X, multiplier=1.6)
mirna_A = create_diff_exp_connections_norm(mirna_X, multiplier=2.0)
circrna_A = create_diff_exp_connections_norm(circrna_X, multiplier=0.75)
te_A = create_diff_exp_connections_norm(te_X, multiplier=1.5)

isolated sample nodes, isolated gene nodes, mean degree: 
tensor(1) tensor(0) tensor(41.2973)
isolated sample nodes, isolated gene nodes, mean degree: 
tensor(0) tensor(11) tensor(23.6216)
isolated sample nodes, isolated gene nodes, mean degree: 
tensor(1) tensor(0) tensor(14.9865)
isolated sample nodes, isolated gene nodes, mean degree: 
tensor(0) tensor(0) tensor(24.5270)


In [203]:
mirna_mrna_interactions.columns = ['id', 'mirna', 'gene']
mirna_mrna_interactions.write_csv('MDS_data_preprocessed/mirna_mrna_interactions.csv')

In [248]:
gg_A = gg_interactions(mrna_features, mrna_features)
pp_A = pp_interactions(mrna_features, mrna_features)

mrna_features_A = torch.logical_or(gg_A, pp_A).int()

mirna_mrna_interactions = get_mirna_gene_interactions(
    mrna_names=mrna_features,
    mirna_names=mirna_features,
    mirna_mrna_db='MDS_data_preprocessed/mirna_mrna_interactions.csv'
)

torch.Size([500, 500])


In [249]:
mrna_features_A.sum(), mirna_mrna_interactions.sum()

(tensor(548), tensor(0.))

In [251]:
circna_mirna_A = get_interactions(
    interactants1=circrna_features,
    interactants2=mirna_features,
    interactant1_colname='circRNA',
    interactant2_colname='miRNAs',
    interact_df=circrna_mirna_interactions
)

circna_mirna_A.sum()

tensor(0.)

In [252]:
# !! normalize the expression data
ss = MinMaxScaler()
ss = ss.fit(mrna_X[train_mask])
mrna_X = torch.tensor(ss.transform(mrna_X), dtype=torch.float32)

ss = MinMaxScaler()
ss = ss.fit(mirna_X[train_mask])
mirna_X = torch.tensor(ss.transform(mirna_X), dtype=torch.float32)

ss = MinMaxScaler()
ss = ss.fit(circrna_X[train_mask])
circrna_X = torch.tensor(ss.transform(circrna_X), dtype=torch.float32)

ss = MinMaxScaler()
ss = ss.fit(te_X[train_mask])
te_X = torch.tensor(ss.transform(te_X), dtype=torch.float32)

In [258]:
data = pyg.data.HeteroData()

proj_dim = 64

data['mrna'].x = mrna_X
data['mirna'].x = mirna_X
data['circrna'].x = circrna_X
data['te'].x = te_X

data['mrna_feature'].x = torch.ones(mrna_X.shape[1], proj_dim)
data['mirna_feature'].x = torch.ones(mirna_X.shape[1], proj_dim)
data['circrna_feature'].x = torch.ones(circrna_X.shape[1], proj_dim)
data['te_feature'].x = torch.ones(te_X.shape[1], proj_dim)

data.y = torch.tensor(y, dtype=torch.long)

data.omics = ['mrna', 'mirna', 'circrna', 'te']
data.feature_names = ['mrna_feature', 'mirna_feature', 'circrna_feature', 'te_feature']

# sample -> feature edges
data['mrna', 'diff_exp', 'mrna_feature'].edge_index = dense_to_coo(mrna_A)
data['mrna', 'diff_exp', 'mrna_feature'].edge_attributes = dense_to_attributes(mrna_A)
data['mirna', 'diff_exp', 'mirna_feature'].edge_index = dense_to_coo(mirna_A)
data['mirna', 'diff_exp', 'mirna_feature'].edge_attributes = dense_to_attributes(mirna_A)
data['circrna', 'diff_exp', 'circrna_feature'].edge_index = dense_to_coo(circrna_A)
data['circrna', 'diff_exp', 'circrna_feature'].edge_attributes = dense_to_attributes(circrna_A)
data['te', 'diff_exp', 'te_feature'].edge_index = dense_to_coo(te_A)
data['te', 'diff_exp', 'te_feature'].edge_attributes = dense_to_attributes(te_A)

# feature -> feature edges
data['mrna_feature', 'interacts_with', 'mrna_feature'].edge_index = dense_to_coo(mrna_features_A)
# data['mrna_feature', 'interacts_with', 'mirna_feature'].edge_index = dense_to_coo(mirna_mrna_interactions)
# data['circrna_feature', 'interacts_with', 'mirna_feature'].edge_index = dense_to_coo(circna_mirna_A)

# masks
data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

# convert to undirected graph
data = T.ToUndirected()(data)

data.num_relations = len(data.edge_index_dict.keys())
print(data.num_relations)

9


In [256]:
data

HeteroData(
  y=[74],
  omics=[4],
  feature_names=[4],
  train_mask=[51],
  val_mask=[11],
  test_mask=[12],
  num_relations=9,
  mrna={ x=[74, 500] },
  mirna={ x=[74, 500] },
  circrna={ x=[74, 50] },
  te={ x=[74, 250] },
  mrna_feature={ x=[500, 50] },
  mirna_feature={ x=[500, 50] },
  circrna_feature={ x=[50, 50] },
  te_feature={ x=[250, 50] },
  (mrna, diff_exp, mrna_feature)={
    edge_index=[2, 3056],
    edge_attributes=[3056, 1],
  },
  (mirna, diff_exp, mirna_feature)={
    edge_index=[2, 1748],
    edge_attributes=[1748, 1],
  },
  (circrna, diff_exp, circrna_feature)={
    edge_index=[2, 1109],
    edge_attributes=[1109, 1],
  },
  (te, diff_exp, te_feature)={
    edge_index=[2, 1815],
    edge_attributes=[1815, 1],
  },
  (mrna_feature, interacts_with, mrna_feature)={ edge_index=[2, 808] },
  (mrna_feature, rev_diff_exp, mrna)={
    edge_index=[2, 3056],
    edge_attributes=[3056, 1],
  },
  (mirna_feature, rev_diff_exp, mirna)={
    edge_index=[2, 1748],
    edge_attr

In [270]:
model = BiRGAT(
    omic_channels=data.omics,
    feature_names=data.feature_names,
    relations=list(data.edge_index_dict.keys()),
    input_dims=[mrna_X.shape[1], mirna_X.shape[1], circrna_X.shape[1], te_X.shape[1]],
    num_classes=len(np.unique(y)),
    proj_dim=proj_dim,
    hidden_channels=[256, 128, 128, 32],
    heads=4,
    dropout=0.1,
)
# from bipartite_gnn.model import NN_encoders_ATT_integrator

# model = NN_encoders_ATT_integrator(
#     omic_channels=data.omics,
#     feature_names=data.feature_names,
#     relations=list(data.edge_index_dict.keys()),
#     input_dims=[mrna_X.shape[1], mirna_X.shape[1], circrna_X.shape[1], te_X.shape[1]],
#     num_classes=len(np.unique(y)),
#     proj_dim=proj_dim,
#     hidden_channels=[128, 128, 64, 32],
#     heads=4,
#     dropout=0.2,
# )

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=5e-4)

ccounts = torch.unique(data.y[train_mask], return_counts=True)[1]
inv_w = 1.0 / ccounts.float()
class_weights = inv_w / inv_w.sum()

trainer = GNNTrainer(
    model=model,
    loss_fn=torch.nn.CrossEntropyLoss(weight=class_weights),
    optimizer=optimizer,
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=15, min_lr=1e-5),
    params={
        "l1_lambda" : 0.1,
    }
)

trainer.train(
    data=data,
    epochs=300,
    log_interval=1,
)

Saving the best model
Epoch: 001, 
Train Loss: 478.0070, Train Acc: 0.6275, Train F1 Macro: 0.2634, Train F1 Weighted: 0.5113
Val Loss: 1.1310, Val Acc: 0.4545, Val F1 Macro: 0.3333, Val F1 Weighted: 0.4091
Test Loss: 1.1145, Test Acc: 0.4167, Test F1 Macro: 0.2792, Test F1 Weighted: 0.4473
##################################################
Epoch: 002, 
Train Loss: 477.9684, Train Acc: 0.5294, Train F1 Macro: 0.3403, Train F1 Weighted: 0.5152
Val Loss: 1.1141, Val Acc: 0.4545, Val F1 Macro: 0.3300, Val F1 Weighted: 0.4279
Test Loss: 1.1048, Test Acc: 0.2500, Test F1 Macro: 0.1818, Test F1 Weighted: 0.2727
##################################################
Epoch: 003, 
Train Loss: 477.9470, Train Acc: 0.4314, Train F1 Macro: 0.3100, Train F1 Weighted: 0.4299
Val Loss: 1.0987, Val Acc: 0.1818, Val F1 Macro: 0.2333, Val F1 Weighted: 0.1273
Test Loss: 1.1071, Test Acc: 0.2500, Test F1 Macro: 0.2526, Test F1 Weighted: 0.2374
##################################################
Epoch: 004, 
Tr

KeyboardInterrupt: 

In [211]:
model = BiRGAT(
    omic_channels=data.omics,
    feature_names=data.feature_names,
    relations=list(data.edge_index_dict.keys()),
    input_dims=[mrna_X.shape[1], mirna_X.shape[1], circrna_X.shape[1], te_X.shape[1]],
    num_classes=len(np.unique(y)),
    proj_dim=proj_dim,
    hidden_channels=[128, 128, 128, 32],
    heads=4,
    dropout=0.4,
)

model.load_state_dict(torch.load('best_model_risk_keep.pth'))

<All keys matched successfully>

In [212]:
fi = model.proj_layer_feature_importance()

In [213]:
fi

{'mrna': tensor([2.9897, 2.6898, 2.7144, 2.3902, 2.4734, 2.5602, 2.3812, 2.8419, 2.1967,
         2.6515, 2.7321, 2.6498, 2.3615, 2.6483, 2.4876, 2.6540, 2.3104, 2.5678,
         2.6535, 2.2543, 2.6008, 2.6394, 2.3170, 2.1762, 2.4542, 2.6779, 2.9010,
         2.6446, 2.7977, 2.7541, 2.6341, 3.0724, 2.5219, 2.7201, 2.6589, 2.6885,
         2.5837, 2.7242, 2.7224, 2.8136, 2.3838, 2.4177, 2.5673, 2.1765, 2.5339,
         2.7361, 3.0345, 2.8542, 2.4809, 2.5106, 2.4445, 2.6219, 3.0571, 2.3017,
         2.6744, 2.8068, 2.8764, 2.5531, 2.7974, 2.6824, 2.3573, 2.8033, 2.6717,
         2.7145, 2.6314, 2.5842, 2.4466, 2.5532, 2.6428, 2.3869, 2.3146, 3.0473,
         2.7194, 2.8721, 2.2363, 2.5967, 2.4823, 2.9428, 2.9205, 2.8014, 2.6737,
         2.7949, 2.9315, 2.8037, 2.2329, 2.6942, 3.1982, 2.2604, 2.7614, 2.3060,
         2.8009, 2.4991, 2.6224, 2.6932, 2.4751, 2.6681, 2.4983, 2.7595, 2.5010,
         2.9579, 2.5836, 2.5914, 2.2181, 2.6006, 2.6587, 2.5358, 2.2813, 2.3986,
         2.6216, 2.8

In [281]:
indices = torch.sort(fi['mrna'], descending=True).indices[:25]
scores = torch.sort(fi['mrna'], descending=True).values[:25]

indices = indices.tolist()

for i, i_ in enumerate(indices):
    print(mrna_features[i_], ",", scores[i].detach().numpy())

GAK , 3.3848748
ZNF627 , 3.218972
GIT1 , 3.1982214
SLC7A6OS , 3.1890595
TBC1D24 , 3.1267643
ACOT11 , 3.112023
SLC35B2 , 3.094448
C11orf94 , 3.081261
TNFRSF4 , 3.0770233
ABCB4 , 3.07237
MYBPC2 , 3.0638366
OR6F1 , 3.0571468
RP11-195F19.29 , 3.053553
FBXO28 , 3.0473137
RP11-468E2.1 , 3.0358038
SYNPO , 3.0345008
SLC24A5 , 3.0140474
TUFT1 , 3.0122452
C11orf42 , 3.0095193
CHKA , 3.0031643
CBARP , 3.0000155
SLC17A5 , 2.998269
FCMR , 2.9943352
OR2D2 , 2.9897134
MS4A3 , 2.9884028


- YPEL2 - https://www.ncbi.nlm.nih.gov/pmc/articles/PMC9204605/
- FSTL3 - may have a role in leukemogenesis https://www.ncbi.nlm.nih.gov/pmc/articles/PMC9619213/
- GPR2 - may have role in liver cancer https://pubmed.ncbi.nlm.nih.gov/35330739/

In [280]:
indices = torch.sort(fi['mirna'], descending=True).indices[:25]
scores = torch.sort(fi['mirna'], descending=True).values[:25]

indices = indices.tolist()

for i, i_ in enumerate(indices):
    print(mirna_features[i_],",", scores[i].detach().numpy())

hsa-mir-8062 , 3.359297
hsa-mir-1288 , 3.2195134
hsa-mir-572 , 3.2063227
hsa-mir-4683 , 3.1698656
hsa-mir-3713 , 3.1634839
hsa-mir-520B , 3.0988014
hsa-mir-6820 , 3.0897644
hsa-mir-3922 , 3.08941
hsa-mir-92A1 , 3.0843172
hsa-mir-1827 , 3.0829682
hsa-mir-4796 , 3.0779226
hsa-mir-941-5 , 3.0616395
hsa-mir-3183 , 3.054103
hsa-mir-222 , 3.0398107
hsa-mir-6775 , 3.0204422
hsa-mir-5007 , 3.0181549
hsa-mir-6780A , 3.017288
hsa-mir-7704 , 3.0132737
hsa-mir-6515 , 3.0051203
hsa-mir-624 , 2.9981623
hsa-mir-3926-1 , 2.993554
hsa-mir-4536-2 , 2.9885006
hsa-mir-3687-2 , 2.9867337
hsa-mir-648 , 2.9826534
hsa-mir-3529 , 2.9738169


- hsa-miR-5195 : myeloid leukemia https://pubmed.ncbi.nlm.nih.gov/33969901/
- hsa-miR-127 : leukemia, myeloma http://www.ncbi.nlm.nih.gov/pubmed/29402726, http://www.ncbi.nlm.nih.gov/pubmed/18478077, http://www.ncbi.nlm.nih.gov/pubmed/33549034
- hsa-mir-192 : high survival for myelodisplastic syndromes, leukemia, multiple myeloma http://www.ncbi.nlm.nih.gov/pubmed/36803590

In [277]:
circrna.filter(pl.col('circRNA_ID').is_in(['hsa_circ_0004277']))

circRNA_ID,V1884,N58,V630,N60,V1297,NV1428,N82,V940,V2092,V624,V777,V553,V1441,V1921,V538,V1857,V456,NV912,V2089,V1823,V2241,N54,V655,V2110,V839,V125,N83,V637,V712,NV911,V2133,V1742,V1591,V108,V1874,V221,V148,N70,N87,V574,V359,V1337,V883,V1592,V1422,V1708,V1505,V18,V1788,V1776,N84,V1800,V716,N86,V888,V1321,V1279,V1528,V344,N85,V1699,V1456,V1394,V714,V67,V1090,V1860,V406,V1834,V1048,V806,V513,V1565,V1920
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64


In [282]:
indices = torch.sort(fi['circrna'], descending=True).indices[:25]
scores = torch.sort(fi['circrna'], descending=True).values[:25]

indices = indices.tolist()

for i, i_ in enumerate(indices):
    print(circrna_features[i_], ",", scores[i].detach().numpy())

hsa_circ_0000722 , 7.516904
hsa_circ_0000443 , 7.3188314
hsa_circ_0000437 , 6.8263416
hsa_circ_0003922 , 6.6998353
hsa_circ_0004435 , 6.69564
hsa_circ_0001432 , 6.671605
hsa_circ_0000643 , 6.6479
hsa_circ_0000615 , 6.605722
hsa_circ_0001445 , 6.5223246
hsa_circ_0014606 , 6.5170345
hsa_circ_0001769 , 6.4628487
hsa_circ_0001944 , 6.43813
hsa_circ_0001861 , 6.417963
hsa_circ_0000711 , 6.3639126
hsa_circ_0008285 , 6.33384
hsa_circ_0001009 , 6.3094645
hsa_circ_0000567 , 6.3080873
hsa_circ_0000489 , 6.1739764
hsa_circ_0000997 , 6.1532373
hsa_circ_0002484 , 6.1344147
hsa_circ_0005158 , 6.117278
hsa_circ_0001727 , 6.102264
hsa_circ_0006127 , 6.094975
hsa_circ_0000896 , 6.0872188
hsa_circ_0001777 , 6.07992


- Hsa_circ_0012152 : leukemia https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7492294/

In [284]:
indices = torch.sort(fi['te'], descending=True).indices[:25]
scores = torch.sort(fi['te'], descending=True).values[:25]

indices = indices.tolist()

for i, i_ in enumerate(indices):
    print(te_features[i_], ",", scores[i].detach().numpy())

SVA_B , 4.420809
ERV3-16A3_I , 4.2351604
LTR79 , 4.2004027
L1MB6_5 , 4.1614795
L1MB5 , 4.151041
AluYbc3a , 4.1345587
MLT1F1 , 4.106223
LTR2752 , 4.1013064
L1MC5 , 4.096092
MER54A , 4.0955353
AluSc8 , 4.0511327
MLT1G3 , 4.045249
AluSx , 4.0314956
LTR47A2 , 4.0308557
MER57E2 , 4.0034447
MER88 , 3.9956846
LTR65 , 3.9850092
THE1A , 3.981634
MER21C , 3.9687796
L1MC4 , 3.965302
AluYf1 , 3.9607303
LTR71A , 3.9584777
LTR31 , 3.9451468
AluSg1 , 3.9353588
MLT1J , 3.9249043
